# Prophet vs No-Prophet Submission Comparison

This notebook trains the same XGBoost per-journey classifier twice:

- `without_prophet`: journey/event features only.
- `with_prophet`: the same features plus Prophet daily order-volume forecast components joined on each journey's last visible action date.

It writes two Kaggle-ready submission CSVs and a comparison table under `joel/kaggle_sumissions/prophet_comparison/`.

In [7]:
# Uncomment this in a fresh environment if these packages are missing.
# %pip install -q polars pyarrow xgboost prophet scikit-learn pandas numpy

## Configuration

`MAX_BALANCED_ROWS` controls runtime. The default below is fast enough for iteration. Set it to `None` for the full 1:19 balanced training set.

In [8]:
from pathlib import Path
import os
import sys

MAX_BALANCED_ROWS = os.getenv("MAX_BALANCED_ROWS", "250000")
MAX_BALANCED_ROWS = None if MAX_BALANCED_ROWS.lower() in {"none", "full", "all"} else int(MAX_BALANCED_ROWS)
REBUILD_PROPHET = os.getenv("REBUILD_PROPHET", "0") == "1"

for candidate in [Path.cwd(), *Path.cwd().parents]:
    pipeline_dir = candidate / "joel" / "kaggle_sumissions" / "prophet_comparison"
    if (pipeline_dir / "prophet_comparison_pipeline.py").exists():
        sys.path.insert(0, str(pipeline_dir))
        break
else:
    raise FileNotFoundError("Could not find prophet_comparison_pipeline.py")

from prophet_comparison_pipeline import run_comparison

## Run Both Submissions

In [ ]:
results = run_comparison(
    max_balanced_rows=MAX_BALANCED_ROWS,
    rebuild_prophet=REBUILD_PROPHET,
)

results["output_paths"]

Project root: /Users/joelyoon/Documents/git_repo/m148-project
Source submission dir: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4
Output dir: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/prophet_comparison
Max balanced rows: 250000
Balanced train rows: 250000 {'failure': 237500, 'success': 12500}
Loaded Prophet forecast: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/data/prophet_daily_order_forecast.csv
Prophet features: ['prophet_orders_yhat', 'prophet_orders_trend', 'prophet_orders_weekly', 'prophet_orders_yearly', 'prophet_orders_holidays']


## Validation and Submission Summary

In [ ]:
results["metrics"]

,valid_brier,valid_prauc,valid_prediction_mean,variant,include_prophet,n_features,train_rows,submission_mean,submission_std,submission_min,submission_p50,submission_max,output_path,prediction_correlation
0,0.036043,0.41515,0.048956,without_prophet,False,26,250000,0.055614,0.076252,0.002488,0.036149,0.951529,/Users/joelyoon/Documents/git_repo/m148-projec...,0.576043
1,0.034595,0.45723,0.048828,with_prophet,True,31,250000,0.184397,0.134146,0.008393,0.149601,0.961590,/Users/joelyoon/Documents/git_repo/m148-projec...,0.576043


## Prediction Delta

In [ ]:
results["delta_summary"]

,delta
count,123467.000000
mean,0.128783
std,0.109658
min,-0.253962
25%,0.050346
50%,0.098918
75%,0.169823
max,0.801700


In [ ]:
results["prediction_comparison"].head(20)

,id,without_prophet,with_prophet,delta_with_minus_without
0,-1000028517 -1627409742,0.013141,0.335439,0.322298
1,-1000060331 -454101160,0.188379,0.162047,-0.026332
2,-1000140359 1672935802,0.030269,0.140320,0.110051
3,-1000141172 2095873834,0.026389,0.161764,0.135375
4,-1000157262 -1303368175,0.027994,0.220190,0.192197
5,-1000185605 -1997573428,0.065476,0.272883,0.207407
6,-100030883 1839598356,0.061444,0.154087,0.092643
7,-1000327390 1275540122,0.030863,0.324093,0.293229
8,-1000334185 -56202835,0.020139,0.176046,0.155906
9,-1000341887 -1501977714,0.018669,0.195314,0.176645


## Graphs

These are the two main visuals to use when explaining the comparison.

### Validation Metrics

![Validation metric comparison](figures/validation_metric_comparison.png)

### Prophet Components

![Daily Prophet components](../submission4/prophet-plots/daily_orders_components.png)